# How this notebook is laid out
Refer to Section 3.6: Building a Knowledge-Based Agent
https://www.ricopic.one/teaching/notes/engineering-artificial-intelligence/knowledge-and-reasoning/building-a-knowledge-based-agent/#k7


In [1]:
# Verify required packages are available
import sys
print(f"Python version: {sys.version}")

try:
    import numpy as np
    print(f"✓ numpy {np.__version__} is available")
except ImportError as e:
    print(f"✗ numpy not available: {e}")

try:
    import matplotlib.pyplot as plt
    print("✓ matplotlib is available")
except ImportError as e:
    print(f"✗ matplotlib not available: {e}")

print("All required packages are ready!")

Python version: 3.12.12 (main, Oct  9 2025, 11:07:00) [Clang 17.0.0 (clang-1700.6.3.2)]
✓ numpy 2.4.1 is available
✓ matplotlib is available
All required packages are ready!


# What is Z3?
Z3 is an open-source SMT (Satisfiability Modulo Theories) solver developed by Microsoft Research. It is an industrial-strength tool used in software verification, security analysis, and AI research. It supports:

Propositional logic with native biconditionals (no manual CNF conversion)
First-order logic with quantifiers (, ) over user-defined sorts
Arithmetic, bit-vectors, arrays, and other theories
We will use Z3 as our reasoning engine throughout this chapter. In this section we use its propositional logic capabilities; in Section 3.7: Building a FOL Agent with Z3 we extend to first-order logic.

In [4]:
# Check if z3-solver is already installed
try:
    import z3
    print(f"✓ z3-solver is already available! Version info: {z3.get_version_string()}")
except ImportError:
    print("✗ z3-solver not available - trying installation with break-system-packages...")
    
    import subprocess
    import sys
    
    # Try with --break-system-packages and --user for safety
    try:
        print("Installing z3-solver with --break-system-packages --user...")
        result = subprocess.run([sys.executable, "-m", "pip", "install", "--break-system-packages", "--user", "z3-solver"], 
                              capture_output=True, text=True, timeout=300)
        if result.returncode == 0:
            print("✓ z3-solver installed successfully!")
            print("Refreshing imports...")
            
            # Invalidate import caches and try importing
            import importlib
            importlib.invalidate_caches()
            
            # Restart the kernel might be needed, but let's try importing first
            try:
                import z3
                print(f"✓ z3-solver imported successfully! Version: {z3.get_version_string()}")
            except ImportError:
                print("⚠️  z3-solver installed but import failed - kernel restart may be needed")
                print("Try restarting the kernel and running this cell again")
        else:
            print(f"✗ Installation failed: {result.stderr}")
            print("\nAlternative solutions:")
            print("1. Install via terminal: python3 -m pip install --break-system-packages --user z3-solver")
            print("2. Use conda if available: conda install z3-solver")
            print("3. Try: brew install z3-solver (if using Homebrew)")
    except subprocess.TimeoutExpired:
        print("✗ Installation timed out (>5 minutes)")
        print("z3-solver can take a long time to compile on some systems")
    except Exception as e:
        print(f"✗ Unexpected error: {e}")

✗ z3-solver not available - trying installation with break-system-packages...
Installing z3-solver with --break-system-packages --user...
✓ z3-solver installed successfully!
Refreshing imports...
✓ z3-solver imported successfully! Version: 4.15.8


In [5]:
# Quick test to verify z3-solver is working correctly
import z3

# Create a simple SAT problem to test
x, y = z3.Ints('x y')
solver = z3.Solver()
solver.add(x + y == 10)
solver.add(x > 5)

if solver.check() == z3.sat:
    model = solver.model()
    print("✓ z3-solver is working correctly!")
    print(f"Example solution: x={model[x]}, y={model[y]}")
    print("You can now proceed with your theorem proving tasks!")
else:
    print("✗ z3-solver test failed")

✓ z3-solver is working correctly!
Example solution: x=6, y=4
You can now proceed with your theorem proving tasks!


In [7]:
# Another quick example to verify the installation

from z3 import Bool, Bools, Solver, And, Or, Not
P, Q = Bools('P Q')
s = Solver()
s.add(P == Q)    # Biconditional --- native, no CNF needed
s.add(P)
print(s.check())  # sat
print(s.model())   # [Q = True, P = True]

sat
[Q = True, P = True]


In [6]:
# Setup the Hazardous Warehouse environment
# See section 3.2.10

"""
Hazardous Warehouse Environment
A partially observable environment for knowledge-based reasoning agents.
The robot must navigate a grid with hidden hazards (damaged floor, malfunctioning
forklift), retrieve a package, and exit safely.
"""
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import NamedTuple
import random
# -----------------------------------------------------------------------------
# Types and Constants
# -----------------------------------------------------------------------------
class Direction(Enum):
    """Cardinal directions the robot can face."""
    NORTH = auto()
    EAST = auto()
    SOUTH = auto()
    WEST = auto()
    def turn_left(self) -> "Direction":
        """Rotate 90° counterclockwise."""
        order = [Direction.NORTH, Direction.WEST, Direction.SOUTH, Direction.EAST]
        return order[(order.index(self) + 1) % 4]
    def turn_right(self) -> "Direction":
        """Rotate 90° clockwise."""
        order = [Direction.NORTH, Direction.EAST, Direction.SOUTH, Direction.WEST]
        return order[(order.index(self) + 1) % 4]
    def delta(self) -> tuple[int, int]:
        """Return (dx, dy) for moving in this direction."""
        deltas = {
            Direction.NORTH: (0, 1),
            Direction.EAST: (1, 0),
            Direction.SOUTH: (0, -1),
            Direction.WEST: (-1, 0),
        }
        return deltas[self]
class Action(Enum):
    """Actions available to the robot."""
    FORWARD = auto()
    TURN_LEFT = auto()
    TURN_RIGHT = auto()
    GRAB = auto()
    SHUTDOWN = auto()
    EXIT = auto()
class Percept(NamedTuple):
    """Sensory information available to the robot at each step."""
    creaking: bool      # Adjacent to damaged floor
    rumbling: bool      # Adjacent to malfunctioning forklift
    beacon: bool        # At package location
    bump: bool          # Hit a wall on last move
    beep: bool          # Shutdown device successfully disabled forklift
# -----------------------------------------------------------------------------
# Robot and Environment State
# -----------------------------------------------------------------------------
@dataclass
class RobotState:
    """Internal state of the robot."""
    x: int
    y: int
    direction: Direction
    has_package: bool = False
    has_shutdown_device: bool = True
    alive: bool = True
@dataclass
class HazardousWarehouseEnv:
    """
    Hazardous Warehouse environment for knowledge-based agents.
    A 4x4 grid (default) where:
    - The robot starts at (1, 1) facing east
    - Some squares have damaged floor (deadly, cause creaking in adjacent squares)
    - One square has a malfunctioning forklift (deadly, causes rumbling in adjacent)
    - One square has the package (emits beacon signal when robot is there)
    - The robot has a one-use shutdown device to disable the forklift
    Coordinates: (x, y) where x is column (1-4), y is row (1-4), origin bottom-left.
    """
    width: int = 4
    height: int = 4
    num_damaged: int = 2
    seed: int | None = None
    # Hidden world state (not visible to agent)
    _damaged: set[tuple[int, int]] = field(default_factory=set)
    _forklift: tuple[int, int] | None = None
    _forklift_alive: bool = True
    _package: tuple[int, int] | None = None
    # Robot state
    _robot: RobotState = field(default_factory=lambda: RobotState(1, 1, Direction.EAST))
    # Episode tracking
    _steps: int = 0
    _total_reward: float = 0.0
    _last_percept: Percept = field(default_factory=lambda: Percept(False, False, False, False, False))
    _terminated: bool = False
    _success: bool = False
    # History for replay
    _history: list[dict] = field(default_factory=list)
    def __post_init__(self):
        self.reset()
    def reset(self, seed: int | None = None) -> Percept:
        """Reset the environment to initial state with new random hazard placement."""
        if seed is not None:
            self.seed = seed
        if self.seed is not None:
            random.seed(self.seed)
        # Generate valid positions (exclude starting square)
        all_positions = [
            (x, y)
            for x in range(1, self.width + 1)
            for y in range(1, self.height + 1)
            if (x, y) != (1, 1)
        ]
        random.shuffle(all_positions)
        # Place hazards
        self._damaged = set(all_positions[:self.num_damaged])
        remaining = all_positions[self.num_damaged:]
        self._forklift = remaining[0]
        self._package = remaining[1]
        self._forklift_alive = True
        # Reset robot
        self._robot = RobotState(1, 1, Direction.EAST)
        # Reset tracking
        self._steps = 0
        self._total_reward = 0.0
        self._terminated = False
        self._success = False
        self._history = []
        # Get initial percept
        self._last_percept = self._get_percept(bump=False, beep=False)
        self._record_state()
        return self._last_percept
    def step(self, action: Action) -> tuple[Percept, float, bool, dict]:
        """
        Execute an action and return (percept, reward, terminated, info).
        Rewards:
        - +1000 for successful exit with package
        - -1000 for death (damaged floor or forklift collision)
        - -1 for each action
        - -10 for using shutdown device
        """
        if self._terminated:
            return self._last_percept, 0.0, True, {"error": "Episode already terminated"}
        reward = -1.0  # Base action cost
        bump = False
        beep = False
        info: dict = {"action": action.name}
        if action == Action.FORWARD:
            bump = self._move_forward()
            if not bump and self._robot.alive:
                # Check for death
                pos = (self._robot.x, self._robot.y)
                if pos in self._damaged:
                    self._robot.alive = False
                    reward = -1000.0
                    self._terminated = True
                    info["death"] = "damaged_floor"
                elif pos == self._forklift and self._forklift_alive:
                    self._robot.alive = False
                    reward = -1000.0
                    self._terminated = True
                    info["death"] = "forklift"
        elif action == Action.TURN_LEFT:
            self._robot.direction = self._robot.direction.turn_left()
        elif action == Action.TURN_RIGHT:
            self._robot.direction = self._robot.direction.turn_right()
        elif action == Action.GRAB:
            pos = (self._robot.x, self._robot.y)
            if pos == self._package and not self._robot.has_package:
                self._robot.has_package = True
                info["grabbed"] = True
            else:
                info["grabbed"] = False
        elif action == Action.SHUTDOWN:
            if self._robot.has_shutdown_device:
                self._robot.has_shutdown_device = False
                reward -= 9.0  # Additional -10 total for shutdown
                beep = self._fire_shutdown()
                info["shutdown_success"] = beep
            else:
                info["shutdown_success"] = False
                info["error"] = "No shutdown device"
        elif action == Action.EXIT:
            pos = (self._robot.x, self._robot.y)
            if pos == (1, 1):
                self._terminated = True
                if self._robot.has_package:
                    reward = 1000.0
                    self._success = True
                    info["exit"] = "success"
                else:
                    info["exit"] = "no_package"
            else:
                info["exit"] = "wrong_location"
        self._steps += 1
        self._total_reward += reward
        # Get new percept
        if self._robot.alive:
            self._last_percept = self._get_percept(bump=bump, beep=beep)
        else:
            self._last_percept = Percept(False, False, False, bump, beep)
        self._record_state(action)
        return self._last_percept, reward, self._terminated, info
    def _move_forward(self) -> bool:
        """Attempt to move forward. Returns True if bumped into wall."""
        dx, dy = self._robot.direction.delta()
        new_x = self._robot.x + dx
        new_y = self._robot.y + dy
        # Check bounds
        if new_x < 1 or new_x > self.width or new_y < 1 or new_y > self.height:
            return True  # Bump
        self._robot.x = new_x
        self._robot.y = new_y
        return False
    def _fire_shutdown(self) -> bool:
        """Fire shutdown device in facing direction. Returns True if forklift hit."""
        if not self._forklift_alive or self._forklift is None:
            return False
        dx, dy = self._robot.direction.delta()
        x, y = self._robot.x, self._robot.y
        # Trace line in facing direction
        while True:
            x += dx
            y += dy
            if x < 1 or x > self.width or y < 1 or y > self.height:
                break
            if (x, y) == self._forklift:
                self._forklift_alive = False
                return True
        return False
    def _get_percept(self, bump: bool, beep: bool) -> Percept:
        """Generate percept for current position."""
        pos = (self._robot.x, self._robot.y)
        adjacent = self._get_adjacent(pos)
        creaking = any(adj in self._damaged for adj in adjacent)
        rumbling = self._forklift_alive and self._forklift in adjacent
        beacon = pos == self._package and not self._robot.has_package
        return Percept(
            creaking=creaking,
            rumbling=rumbling,
            beacon=beacon,
            bump=bump,
            beep=beep,
        )
    def _get_adjacent(self, pos: tuple[int, int]) -> list[tuple[int, int]]:
        """Return list of adjacent positions (cardinal directions only)."""
        x, y = pos
        candidates = [(x-1, y), (x+1, y), (x, y-1), (x, y+1)]
        return [
            (ax, ay) for ax, ay in candidates
            if 1 <= ax <= self.width and 1 <= ay <= self.height
        ]
    def _record_state(self, action: Action | None = None) -> None:
        """Record current state for replay/visualization."""
        self._history.append({
            "step": self._steps,
            "action": action.name if action else None,
            "robot_x": self._robot.x,
            "robot_y": self._robot.y,
            "direction": self._robot.direction.name,
            "has_package": self._robot.has_package,
            "has_shutdown": self._robot.has_shutdown_device,
            "alive": self._robot.alive,
            "forklift_alive": self._forklift_alive,
            "percept": self._last_percept._asdict(),
            "total_reward": self._total_reward,
        })
    # -------------------------------------------------------------------------
    # Public query methods for agents
    # -------------------------------------------------------------------------
    @property
    def robot_position(self) -> tuple[int, int]:
        """Current robot position (x, y)."""
        return (self._robot.x, self._robot.y)
    @property
    def robot_direction(self) -> Direction:
        """Current robot facing direction."""
        return self._robot.direction
    @property
    def has_package(self) -> bool:
        """Whether robot is carrying the package."""
        return self._robot.has_package
    @property
    def has_shutdown_device(self) -> bool:
        """Whether robot still has the shutdown device."""
        return self._robot.has_shutdown_device
    @property
    def is_alive(self) -> bool:
        """Whether robot is still operational."""
        return self._robot.alive
    @property
    def steps(self) -> int:
        """Number of steps taken."""
        return self._steps
    @property
    def total_reward(self) -> float:
        """Cumulative reward."""
        return self._total_reward
    @property
    def history(self) -> list[dict]:
        """Episode history for replay."""
        return self._history.copy()
    # -------------------------------------------------------------------------
    # Methods for visualization (reveal hidden state)
    # -------------------------------------------------------------------------
    def get_true_state(self) -> dict:
        """Return complete world state (for visualization/debugging only)."""
        return {
            "width": self.width,
            "height": self.height,
            "damaged": list(self._damaged),
            "forklift": self._forklift,
            "forklift_alive": self._forklift_alive,
            "package": self._package,
            "robot": {
                "x": self._robot.x,
                "y": self._robot.y,
                "direction": self._robot.direction.name,
                "has_package": self._robot.has_package,
                "has_shutdown": self._robot.has_shutdown_device,
                "alive": self._robot.alive,
            },
            "terminated": self._terminated,
            "success": self._success,
        }
    def render(self, reveal: bool = False) -> str:
        """
        Render the grid as ASCII.
        If reveal=False (default), only shows what robot has visited.
        If reveal=True, shows complete world state.
        """
        lines = []
        # Header with column numbers
        lines.append("  " + " ".join(str(x) for x in range(1, self.width + 1)))
        for y in range(self.height, 0, -1):
            row = [str(y)]
            for x in range(1, self.width + 1):
                pos = (x, y)
                if pos == (self._robot.x, self._robot.y):
                    if not self._robot.alive:
                        row.append("X")
                    elif self._robot.has_package:
                        row.append("@")  # Robot with package
                    else:
                        # Show direction
                        arrows = {
                            Direction.NORTH: "^",
                            Direction.EAST: ">",
                            Direction.SOUTH: "v",
                            Direction.WEST: "<",
                        }
                        row.append(arrows[self._robot.direction])
                elif reveal:
                    if pos in self._damaged:
                        row.append("D")
                    elif pos == self._forklift:
                        row.append("F" if self._forklift_alive else "f")
                    elif pos == self._package and not self._robot.has_package:
                        row.append("P")
                    else:
                        row.append(".")
                else:
                    row.append("?")
            lines.append(" ".join(row))
        return "\n".join(lines)
# -----------------------------------------------------------------------------
# Example Usage
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    # Create environment with fixed seed for reproducibility
    env = HazardousWarehouseEnv(seed=42)
    print("=== Hazardous Warehouse Environment ===")
    print("\nTrue state (hidden from agent):")
    print(env.render(reveal=True))
    print("\nAgent's view:")
    print(env.render(reveal=False))
    print(f"\nInitial percept: {env._last_percept}")
    print(f"Robot at: {env.robot_position}, facing {env.robot_direction.name}")
    # Take a few actions
    actions = [Action.FORWARD, Action.TURN_LEFT, Action.FORWARD]
    for action in actions:
        percept, reward, done, info = env.step(action)
        print(f"\nAction: {action.name}")
        print(f"Percept: {percept}")
        print(f"Reward: {reward}, Done: {done}")
        print(f"Position: {env.robot_position}, Facing: {env.robot_direction.name}")
    print("\n=== Final State ===")
    print(env.render(reveal=True))

=== Hazardous Warehouse Environment ===

True state (hidden from agent):
  1 2 3 4
4 . P . .
3 . . . D
2 . . D .
1 > . F .

Agent's view:
  1 2 3 4
4 ? ? ? ?
3 ? ? ? ?
2 ? ? ? ?
1 > ? ? ?

Initial percept: Percept(creaking=False, rumbling=False, beacon=False, bump=False, beep=False)
Robot at: (1, 1), facing EAST

Action: FORWARD
Percept: Percept(creaking=False, rumbling=True, beacon=False, bump=False, beep=False)
Reward: -1.0, Done: False
Position: (2, 1), Facing: EAST

Action: TURN_LEFT
Percept: Percept(creaking=False, rumbling=True, beacon=False, bump=False, beep=False)
Reward: -1.0, Done: False
Position: (2, 1), Facing: NORTH

Action: FORWARD
Percept: Percept(creaking=True, rumbling=False, beacon=False, bump=False, beep=False)
Reward: -1.0, Done: False
Position: (2, 2), Facing: NORTH

=== Final State ===
  1 2 3 4
4 . P . .
3 . . . D
2 . ^ D .
1 . . F .


In [8]:
# Z3's Solver maintains a set of assertions. The add() method implements TELL from Section 3.1: Knowledge-Based Agents and the Limits of Search:

solver = Solver()
solver.add(P == Q)   # TELL: P <=> Q
solver.add(P)         # TELL: P is true

# For ASK, we need entailment checking: does the KB entail ? We use the refutation method—if  is unsatisfiable, then  must follow from the KB. 
# Z3's push() and pop() methods create and restore checkpoints on the assertion stack, making this clean:

from z3 import Not, unsat
def z3_entails(solver, query):
    """Check whether the solver's current assertions entail query."""
    solver.push()
    solver.add(Not(query))
    result = solver.check() == unsat
    solver.pop()
    return result

# If check() returns unsat, no interpretation can make the KB true while making  false—so  must follow from the KB. 
# The pop() restores the solver to its state before the query, leaving the KB unchanged.



# Encoding the Warehouse
To use propositional logic, we need propositional symbols for every relevant fact about the warehouse. We adopt the following naming convention:
 
Propositional symbols for the Hazardous Warehouse
Symbol	Meaning
D_x_y	Damaged floor at 
F_x_y	Forklift at 
C_x_y	Creaking perceived at 
R_x_y	Rumbling perceived at 
OK_x_y	Square  is safe to enter

In [9]:
# We define helper functions that return Z3 Bool variables:
from z3 import Bool
def damaged(x, y):
    return Bool(f'D_{x}_{y}')
def forklift_at(x, y):
    return Bool(f'F_{x}_{y}')
def creaking_at(x, y):
    return Bool(f'C_{x}_{y}')
def rumbling_at(x, y):
    return Bool(f'R_{x}_{y}')
def safe(x, y):
    return Bool(f'OK_{x}_{y}')

# For example, damaged(3, 1) returns the Z3 Bool variable D_3_1, and Not(damaged(3, 1)) returns its negation. 
# Z3 interns variables by name: calling Bool('D_3_1') twice returns the same object, so these helpers can be called freely without creating duplicates.
# We also need an adjacency helper. In the  grid, a square's neighbors are the squares one step away in each cardinal direction (north, south, east, west), excluding squares outside the grid:
def get_adjacent(x, y, width=4, height=4):
    result = []
    for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nx, ny = x + dx, y + dy
        if 1 <= nx <= width and 1 <= ny <= height:
            result.append((nx, ny))
    return result

# 3.6.4 Encoding Warehouse Physics
Now we populate the knowledge base with the rules that govern the warehouse. These are the same rules we wrote in propositional logic in Section 3.3.10: A Propositional KB for the Hazardous Warehouse, but now expressed in Python using Z3.


In [10]:
# 3.6.4.1 Creaking Rules
# For each square , creaking is perceived there if and only if at least one adjacent square has damaged floor. As a biconditional:
# C_{x,y} <=> (D_{a1,b1} OR D_{a2,b2} OR ...)
# where (a1, b1), (a2, b2), ... are the squares adjaced to (x,y)

# In Z3, this is a single line per square:
adj = get_adjacent(2, 1)  # [(1,1), (3,1), (2,2)]
solver.add(creaking_at(2, 1) == Or([damaged(a, b) for a, b in adj]))
# C_2_1 == Or(D_1_1, D_3_1, D_2_2)

# The == operator creates a biconditional directly. No associate helper, no string-based <=> operator, no manual CNF conversion.

In [11]:
# 3.6.4.2 Rumbling Rules
# The rumbling rules are identical in structure, with the forklift playing the role of damaged floor:
# R{x,y} <=> (F_{a1,b1} OR F_{a2,b2} OR ...)

solver.add(rumbling_at(2, 1) == Or([forklift_at(a, b) for a, b in adj]))

In [12]:
# 3.6.4.3 Safety Rules
# A square is safe if and only if it has no damaged floor and no forklift:
# OK_{x,y} <=> (NOT D_{x,y} AND NOT F_{x,y})

solver.add(safe(2, 1) == And(Not(damaged(2, 1)), Not(forklift_at(2, 1))))

In [13]:
# 3.6.4.4 Putting It All Together
# The function build_warehouse_kb creates a Solver and encodes all the rules for every square, plus the initial knowledge that  is safe:

from z3 import Solver, Or, And, Not
def build_warehouse_kb(width=4, height=4):
    solver = Solver()
    # The starting square is safe.
    solver.add(Not(damaged(1, 1)))
    solver.add(Not(forklift_at(1, 1)))
    for x in range(1, width + 1):
        for y in range(1, height + 1):
            adj = get_adjacent(x, y, width, height)
            # Creaking iff damaged adjacent
            solver.add(creaking_at(x, y) == Or([damaged(a, b) for a, b in adj]))
            # Rumbling iff forklift adjacent
            solver.add(rumbling_at(x, y) == Or([forklift_at(a, b) for a, b in adj]))
            # Safety rule
            solver.add(
                safe(x, y) == And(Not(damaged(x, y)), Not(forklift_at(x, y)))
            )
    return solver

# Each biconditional reads almost exactly like the mathematical formula:
# C{x,y} <=> (D_{a,b} OR D_{c,d} OR ...)
# becomes creaking_at(x, y) == Or([damaged(a, b) for ...]). Z3 handles the internal conversion to a form it can reason about efficiently.
# After calling build_warehouse_kb(), the solver contains the complete physics of the warehouse. It knows nothing yet about what the robot has perceived—that comes next.

In [14]:
# 3.6.5 Telling Percepts
# Each time the robot visits a square, it receives a Percept from the environment with boolean fields creaking and rumbling (among others). 
# We translate these directly into Z3 assertions:

def tell_percepts(solver, percept, x, y):
    """TELL the solver the percepts observed at (x, y)."""
    if percept.creaking:
        solver.add(creaking_at(x, y))
    else:
        solver.add(Not(creaking_at(x, y)))
    if percept.rumbling:
        solver.add(rumbling_at(x, y))
    else:
        solver.add(Not(rumbling_at(x, y)))

# Both the positive and negative cases matter. 
# Telling the solver Not(creaking_at(2, 1)) (no creaking at ) is just as important as telling it creaking_at(2, 1)—the absence of a percept is information.

In [15]:
# 3.6.6 Asking About Safety
# With the physics encoded and percepts told, we can ASK the solver whether a square is safe:

z3_entails(solver, safe(2, 1))        # True if solver entails OK_2_1
z3_entails(solver, Not(safe(3, 1)))   # True if solver entails ~OK_3_1

# There are three possible outcomes for any square:

# Provably safe: z3_entails(solver, safe(x, y)) returns True. The agent can enter safely.
# Provably dangerous: z3_entails(solver, Not(safe(x, y))) returns True. The agent must avoid it.
# Unknown: Neither query returns True. The solver does not have enough information. The agent should be cautious and avoid the square until more evidence is available.

False

3.6.6.1 Manual Walkthrough
Let us trace through the first few steps of the example layout from Section 3.2: 
The Hazardous Warehouse Environment (damaged floor at  and , forklift at , package at ).

In [17]:
# Step 1: At (1,1) , perceiving no creaking, no rumbling.

solver = build_warehouse_kb()
tell_percepts(solver, Percept(creaking=False, rumbling=False,
                              beacon=False, bump=False, beep=False), 1, 1)
# ASK about adjacent squares
print(z3_entails(solver, safe(2, 1)))  # True
print(z3_entails(solver, safe(1, 2)))  # True

# No creaking at (1,1) means no adjacent square has damaged floor. No rumbling means no adjacent square has the forklift. 
# The solver derives that both (2,1) and (1,2) are safe—exactly the reasoning from Section 3.2: The Hazardous Warehouse Environment.

True
True


In [18]:
# Step 2: Move to (2,1), perceiving creaking but no rumbling.

tell_percepts(solver, Percept(creaking=True, rumbling=False,
                              beacon=False, bump=False, beep=False), 2, 1)
print(z3_entails(solver, safe(3, 1)))        # False (unknown)
print(z3_entails(solver, Not(safe(3, 1))))   # False (unknown)
print(z3_entails(solver, safe(2, 2)))        # False (unknown)

# Creaking at (2,1) means damaged floor at (1,1), (3,1), or (2,2). 
# Since (1,1)  is known safe, the damage is at (3,1) or (2,2) — but the solver cannot yet determine which. Both remain unknown.

False
False
False


In [19]:
# Step 3: Visit (1,2) , perceiving rumbling but no creaking.

tell_percepts(solver, Percept(creaking=False, rumbling=True,
                              beacon=False, bump=False, beep=False), 1, 2)
print(z3_entails(solver, safe(2, 2)))        # True!
print(z3_entails(solver, Not(safe(3, 1))))   # True!
print(z3_entails(solver, Not(safe(1, 3))))   # True!

# No creaking at (1,2) rules out damaged floor at (2,2). 
# Combined with the earlier creaking at (2,1), the solver now deduces that (3,1)  must have damaged floor. 
# Rumbling at (1,2) combined with no rumbling at (2,1) identifies the forklift at (1,3). 
# The chain of inference unfolds automatically—the same reasoning we did by hand in Section 3.2: The Hazardous Warehouse Environment, but performed mechanically by the solver.

True
True
True


#3.6.7 The Agent Loop
With the solver machinery in place, the agent operates in a loop:

1. TELL the solver about percepts at the current location.
2. ASK the solver to classify every unknown square as safe, dangerous, or still unknown. Update the known_safe and known_dangerous sets.
3. Choose an action:
    - If the beacon is detected, GRAB the package.
    - If carrying the package, plan a path through safe squares to  and EXIT.
    - Otherwise, plan a path to the nearest safe unvisited square and move there.
    - If no safe unvisited square is reachable, return to  and exit.
4. Execute the action, updating position and direction.
5. Repeat until the episode ends.

In [20]:
# 3.6.7.1 Path Planning
# The agent plans paths using breadth-first search (BFS) through the known_safe set. This guarantees the shortest path through squares the agent has proven safe:

from collections import deque
def plan_path(start, goal_set, known_safe, width, height):
    """BFS from start to any cell in goal_set, moving only through known_safe."""
    queue = deque([(start, [start])])
    seen = {start}
    while queue:
        (cx, cy), path = queue.popleft()
        if (cx, cy) in goal_set:
            return path
        for nx, ny in get_adjacent(cx, cy, width, height):
            if (nx, ny) not in seen and (nx, ny) in known_safe:
                seen.add((nx, ny))
                queue.append(((nx, ny), path + [(nx, ny)]))
    return None  # No path found

In [21]:
# 3.6.7.2 Converting Paths to Actions
# The environment accepts actions like FORWARD, TURN_LEFT, and TURN_RIGHT. To follow a path, the agent must convert each step into a sequence of turns 
# (to face the right direction) followed by a forward move.

# For efficient turning, we compute whether turning left or right is shorter:

def turns_between(current, target):
    """Return the shortest sequence of turn actions from current to target direction."""
    if current == target:
        return []
    # Count steps in each direction and choose the shorter one.
    ...

# The full implementation handles this correctly by indexing into the ordered list of directions (NORTH, EAST, SOUTH, WEST) and comparing clockwise vs. counter-clockwise distances.

In [25]:
# 3.6.8 Complete Implementation
# The following module contains the complete knowledge-based agent. It brings together all the pieces: Z3 entailment checking, variable helpers, physics encoding, percept telling, safety querying, path planning, and the decision loop.

"""
Knowledge-Based Agent for the Hazardous Warehouse (Propositional Z3)
Uses Z3's SMT solver with grounded propositional variables to reason
about safety and navigate the warehouse to retrieve the package.
This agent implements the TELL/ASK loop:
  1. TELL the solver about percepts (solver.add)
  2. ASK via entailment check (push/Not(query)/check/pop)
  3. Plan a path through safe squares toward the goal
  4. Execute actions and repeat
The knowledge base encodes the physics of the warehouse using one Bool
variable per square per predicate:
  - Creaking at (x,y) iff damaged floor in an adjacent square
  - Rumbling at (x,y) iff forklift in an adjacent square
  - A square is safe iff it has no damaged floor and no forklift
"""
from collections import deque
from z3 import Bool, Or, And, Not, Solver, unsat
# Classes are already defined in previous cells - no import needed

# ---------------------------------------------------------------------------
# Z3 Entailment Check
# ---------------------------------------------------------------------------
def z3_entails(solver, query):
    """Check whether the solver's current assertions entail *query*.
    Uses the refutation method: push a checkpoint, assert Not(query),
    and check satisfiability.  If unsat, the negated query is
    inconsistent with the KB --- meaning the KB entails the query.
    Pop restores the solver to its previous state.
    """
    solver.push()
    solver.add(Not(query))
    result = solver.check() == unsat
    solver.pop()
    return result

# ---------------------------------------------------------------------------
# Propositional Variable Helpers
# ---------------------------------------------------------------------------
def damaged(x, y):
    """Z3 Bool variable: damaged floor at (x, y)."""
    return Bool(f'D_{x}_{y}')

def forklift_at(x, y):
    """Z3 Bool variable: forklift at (x, y)."""
    return Bool(f'F_{x}_{y}')

def creaking_at(x, y):
    """Z3 Bool variable: creaking perceived at (x, y)."""
    return Bool(f'C_{x}_{y}')

def rumbling_at(x, y):
    """Z3 Bool variable: rumbling perceived at (x, y)."""
    return Bool(f'R_{x}_{y}')

def safe(x, y):
    """Z3 Bool variable: square (x, y) is safe to enter."""
    return Bool(f'OK_{x}_{y}')

# ---------------------------------------------------------------------------
# Adjacency
# ---------------------------------------------------------------------------
def get_adjacent(x, y, width=4, height=4):
    """Return the list of (x, y) positions adjacent to (x, y)."""
    result = []
    for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nx, ny = x + dx, y + dy
        if 1 <= nx <= width and 1 <= ny <= height:
            result.append((nx, ny))
    return result

# ---------------------------------------------------------------------------
# Knowledge-Base Construction
# ---------------------------------------------------------------------------
def build_warehouse_kb(width=4, height=4):
    """Build a Z3 Solver populated with the physics of the warehouse.
    The solver contains three kinds of constraints for every square (x, y):
    1. Creaking biconditional
       C_x_y == Or(D_a1_b1, D_a2_b2, ...)
       where (a_i, b_i) are the squares adjacent to (x, y).
    2. Rumbling biconditional
       R_x_y == Or(F_a1_b1, F_a2_b2, ...)
    3. Safety biconditional
       OK_x_y == And(Not(D_x_y), Not(F_x_y))
    Z3's native == operator handles biconditionals directly ---
    no manual CNF conversion is needed.
    It also encodes the initial knowledge that the starting square (1, 1)
    has no damaged floor and no forklift.
    """
    solver = Solver()
    # The starting square is safe.
    solver.add(Not(damaged(1, 1)))
    solver.add(Not(forklift_at(1, 1)))
    for x in range(1, width + 1):
        for y in range(1, height + 1):
            adj = get_adjacent(x, y, width, height)
            # --- Creaking rule ---
            solver.add(creaking_at(x, y) == Or([damaged(a, b) for a, b in adj]))
            # --- Rumbling rule ---
            solver.add(rumbling_at(x, y) == Or([forklift_at(a, b) for a, b in adj]))
            # --- Safety rule ---
            solver.add(
                safe(x, y) == And(Not(damaged(x, y)), Not(forklift_at(x, y)))
            )
    return solver

# ---------------------------------------------------------------------------
# Turning Helpers
# ---------------------------------------------------------------------------
_DIRECTION_ORDER = [Direction.NORTH, Direction.EAST, Direction.SOUTH, Direction.WEST]

def _direction_index(d):
    return _DIRECTION_ORDER.index(d)

def turns_between(current, target):
    """Return a list of TURN_LEFT / TURN_RIGHT actions to face *target*.
    Chooses the shortest rotation direction.
    """
    if current == target:
        return []
    ci = _direction_index(current)
    ti = _direction_index(target)
    right_steps = (ti - ci) % 4   # clockwise
    left_steps = (ci - ti) % 4    # counter-clockwise
    if right_steps <= left_steps:
        return [Action.TURN_RIGHT] * right_steps
    else:
        return [Action.TURN_LEFT] * left_steps

def delta_to_direction(dx, dy):
    """Map a movement delta to the Direction enum."""
    return {
        (0, 1): Direction.NORTH,
        (0, -1): Direction.SOUTH,
        (1, 0): Direction.EAST,
        (-1, 0): Direction.WEST,
    }[(dx, dy)]

# ---------------------------------------------------------------------------
# Knowledge-Based Agent
# ---------------------------------------------------------------------------
class WarehouseKBAgent:
    """A knowledge-based agent for the Hazardous Warehouse.
    The agent maintains:
      - A Z3 Solver with physics rules and accumulated percepts
      - Sets of known-safe and known-dangerous squares
      - A queue of planned actions
      - Its own position, direction, and inventory state
    Decision strategy (in priority order):
      1. If the beacon is detected, GRAB the package.
      2. If carrying the package, navigate to (1,1) and EXIT.
      3. Otherwise, explore the nearest safe unvisited square.
      4. If no safe unvisited square is reachable, return to (1,1) and EXIT.
    """
    def __init__(self, env):
        self.env = env
        self.solver = build_warehouse_kb(env.width, env.height)
        self.x = 1
        self.y = 1
        self.direction = Direction.EAST
        self.has_package = False
        self.visited = {(1, 1)}
        self.known_safe = {(1, 1)}
        self.known_dangerous = set()
        self.action_queue = []
        self.step_count = 0

    # ----- Percepts ----------------------------------------------------------
    def tell_percepts(self, percept):
        """Translate a Percept into Z3 assertions and TELL the solver."""
        x, y = self.x, self.y
        if percept.creaking:
            self.solver.add(creaking_at(x, y))
        else:
            self.solver.add(Not(creaking_at(x, y)))
        if percept.rumbling:
            self.solver.add(rumbling_at(x, y))
        else:
            self.solver.add(Not(rumbling_at(x, y)))

    # ----- Safety queries ----------------------------------------------------
    def update_safety(self):
        """ASK the solver about every square whose status is still unknown."""
        for x in range(1, self.env.width + 1):
            for y in range(1, self.env.height + 1):
                pos = (x, y)
                if pos in self.known_safe or pos in self.known_dangerous:
                    continue
                if z3_entails(self.solver, safe(x, y)):
                    self.known_safe.add(pos)
                elif z3_entails(self.solver, Not(safe(x, y))):
                    self.known_dangerous.add(pos)

    # ----- Path planning -----------------------------------------------------
    def plan_path(self, start, goal_set):
        """BFS through known-safe squares from *start* to any cell in *goal_set*.
        Returns a list of (x, y) positions forming the path (including
        *start* and the reached goal), or None if no path exists.
        """
        queue = deque([(start, [start])])
        seen = {start}
        while queue:
            (cx, cy), path = queue.popleft()
            if (cx, cy) in goal_set:
                return path
            for nx, ny in get_adjacent(cx, cy, self.env.width, self.env.height):
                if (nx, ny) not in seen and (nx, ny) in self.known_safe:
                    seen.add((nx, ny))
                    queue.append(((nx, ny), path + [(nx, ny)]))
        return None

    def path_to_actions(self, path):
        """Convert a position path into a sequence of Actions.
        Returns (actions, final_direction) where *actions* is the list of
        TURN_LEFT / TURN_RIGHT / FORWARD actions and *final_direction* is
        the direction the robot faces after executing them all.
        """
        actions = []
        direction = self.direction
        for i in range(1, len(path)):
            dx = path[i][0] - path[i - 1][0]
            dy = path[i][1] - path[i - 1][1]
            target_dir = delta_to_direction(dx, dy)
            actions.extend(turns_between(direction, target_dir))
            actions.append(Action.FORWARD)
            direction = target_dir
        return actions, direction

    # ----- Decision logic ----------------------------------------------------
    def choose_action(self, percept):
        """Select the next action based on the current state of knowledge."""
        # Execute queued actions first (from a multi-step plan).
        if self.action_queue:
            return self.action_queue.pop(0)
        # 1. If the beacon is on, grab the package.
        if percept.beacon and not self.has_package:
            return Action.GRAB
        # 2. If carrying the package, navigate home and exit.
        if self.has_package:
            if (self.x, self.y) == (1, 1):
                return Action.EXIT
            path = self.plan_path((self.x, self.y), {(1, 1)})
            if path and len(path) > 1:
                actions, _ = self.path_to_actions(path)
                self.action_queue = actions[1:]
                return actions[0]
            # Already at (1,1) or can't find path — just exit.
            return Action.EXIT
        # 3. Explore the nearest safe unvisited square.
        safe_unvisited = self.known_safe - self.visited
        if safe_unvisited:
            path = self.plan_path((self.x, self.y), safe_unvisited)
            if path and len(path) > 1:
                actions, _ = self.path_to_actions(path)
                self.action_queue = actions[1:]
                return actions[0]
        # 4. Nothing left to explore — go home and exit.
        if (self.x, self.y) == (1, 1):
            return Action.EXIT
        path = self.plan_path((self.x, self.y), {(1, 1)})
        if path and len(path) > 1:
            actions, _ = self.path_to_actions(path)
            self.action_queue = actions[1:]
            self.action_queue.append(Action.EXIT)
            return actions[0]
        return Action.EXIT

    # ----- Execution ---------------------------------------------------------
    def execute_action(self, action):
        """Send *action* to the environment and update internal bookkeeping."""
        percept, reward, done, info = self.env.step(action)
        if action == Action.FORWARD and not percept.bump:
            dx, dy = self.direction.delta()
            self.x += dx
            self.y += dy
            self.visited.add((self.x, self.y))
        elif action == Action.TURN_LEFT:
            self.direction = self.direction.turn_left()
        elif action == Action.TURN_RIGHT:
            self.direction = self.direction.turn_right()
        elif action == Action.GRAB and info.get("grabbed"):
            self.has_package = True
        self.step_count += 1
        return percept, reward, done, info

    # ----- Main loop ---------------------------------------------------------
    def run(self, verbose=True):
        """Run the full perceive-tell-ask-act loop until the episode ends."""
        # Process the initial percept at (1, 1).
        percept = self.env._last_percept
        self.tell_percepts(percept)
        self.update_safety()
        if verbose:
            print(f"Start at ({self.x},{self.y}) facing {self.direction.name}")
            print(f"  Percept: {percept}")
            print(f"  Known safe: {sorted(self.known_safe)}")
        while True:
            action = self.choose_action(percept)
            percept, reward, done, info = self.execute_action(action)
            if verbose:
                print(f"\nStep {self.step_count}: {action.name}")
                print(f"  Position: ({self.x},{self.y}), Facing: {self.direction.name}")
                print(f"  Percept: {percept}")
                print(f"  Info: {info}")
            if done:
                if verbose:
                    print(f"\n{'=' * 40}")
                    print(f"Episode ended.  Reward: {self.env.total_reward:.0f}")
                    print(f"Steps taken: {self.step_count}")
                    success = info.get("exit") == "success"
                    print(f"Success: {success}")
                return
            # After moving to a new square, tell percepts and re-query safety.
            if action == Action.FORWARD and not percept.bump:
                self.tell_percepts(percept)
                self.update_safety()
                if verbose:
                    print(f"  Known safe: {sorted(self.known_safe)}")
                    print(f"  Known dangerous: {sorted(self.known_dangerous)}")

# ---------------------------------------------------------------------------
# Main — run on the example layout from the textbook
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # Create environment with seed 0 instead of using external layout configuration
    env = HazardousWarehouseEnv(seed=0)
    print("True state (hidden from the agent):")
    print(env.render(reveal=True))
    print()
    agent = WarehouseKBAgent(env)
    agent.run(verbose=True)

True state (hidden from the agent):
  1 2 3 4
4 . . D .
3 D P F .
2 . . . .
1 > . . .

Start at (1,1) facing EAST
  Percept: Percept(creaking=False, rumbling=False, beacon=False, bump=False, beep=False)
  Known safe: [(1, 1), (1, 2), (2, 1)]

Step 1: FORWARD
  Position: (2,1), Facing: EAST
  Percept: Percept(creaking=False, rumbling=False, beacon=False, bump=False, beep=False)
  Info: {'action': 'FORWARD'}
  Known safe: [(1, 1), (1, 2), (2, 1), (2, 2), (3, 1)]
  Known dangerous: []

Step 2: FORWARD
  Position: (3,1), Facing: EAST
  Percept: Percept(creaking=False, rumbling=False, beacon=False, bump=False, beep=False)
  Info: {'action': 'FORWARD'}
  Known safe: [(1, 1), (1, 2), (2, 1), (2, 2), (3, 1), (3, 2), (4, 1)]
  Known dangerous: []

Step 3: FORWARD
  Position: (4,1), Facing: EAST
  Percept: Percept(creaking=False, rumbling=False, beacon=False, bump=False, beep=False)
  Info: {'action': 'FORWARD'}
  Known safe: [(1, 1), (1, 2), (2, 1), (2, 2), (3, 1), (3, 2), (4, 1), (4, 2)]
  Kno

In [ ]:
# 3.6.9 Running the Agent
# To run the agent on the example layout from Section 3.2: The Hazardous Warehouse Environment:

# Classes are already defined in previous cells - no import needed
# Using HazardousWarehouseEnv from cell 7 and WarehouseKBAgent from cell 25

env = HazardousWarehouseEnv(seed=0)
print("True state (hidden from agent):")
print(env.render(reveal=True))
print()

agent = WarehouseKBAgent(env)
agent.run(verbose=True)

# The agent prints a step-by-step trace. At each step it reports its position, the percepts received, the action taken, and the updated sets of known-safe 
# and known-dangerous squares.

# On the example layout (damaged floor at (3,1) and (3,3), forklift at (1,3), package at (2,3), the agent:

# Starts at (1,1) and deduces that (2,1) and (1,2)  are safe.
# Explores (2,1) (creaking) and (1,2)  (rumbling), deducing that (3,1) is damaged, (1,3) has the forklift, and (2,2) is safe.
# Explores (2,2)  and then (2,3), where it detects the beacon and grabs the package.
# Plans a return path through safe squares to (1,1) and exits.

# The entire process is driven by Z3's satisfiability checking on the propositional KB—the same logical foundations developed in 
# Section 3.3: Propositional Logic and Section 3.3.18: Resolution and Completeness, applied mechanically.

True state (hidden from agent):
  1 2 3 4
4 . . D .
3 D P F .
2 . . . .
1 > . . .

Start at (1,1) facing EAST
  Percept: Percept(creaking=False, rumbling=False, beacon=False, bump=False, beep=False)
  Known safe: [(1, 1), (1, 2), (2, 1)]

Step 1: FORWARD
  Position: (2,1), Facing: EAST
  Percept: Percept(creaking=False, rumbling=False, beacon=False, bump=False, beep=False)
  Info: {'action': 'FORWARD'}
  Known safe: [(1, 1), (1, 2), (2, 1), (2, 2), (3, 1)]
  Known dangerous: []

Step 2: FORWARD
  Position: (3,1), Facing: EAST
  Percept: Percept(creaking=False, rumbling=False, beacon=False, bump=False, beep=False)
  Info: {'action': 'FORWARD'}
  Known safe: [(1, 1), (1, 2), (2, 1), (2, 2), (3, 1), (3, 2), (4, 1)]
  Known dangerous: []

Step 3: FORWARD
  Position: (4,1), Facing: EAST
  Percept: Percept(creaking=False, rumbling=False, beacon=False, bump=False, beep=False)
  Info: {'action': 'FORWARD'}
  Known safe: [(1, 1), (1, 2), (2, 1), (2, 2), (3, 1), (3, 2), (4, 1), (4, 2)]
  Known d

# Assignment Tasks Section

In [5]:
# Assignment task 1:
# Setup and exploration: Import Bool, Bools, Or, And, Not, Solver, unsat from z3, and HazardousWarehouseEnv from hazardous_warehouse_env.py
from z3 import Bool, Bools, Or, And, Not, Solver, unsat

# Create boolean variables P and Q
P = Bool('P')
Q = Bool('Q')

# Create a solver instance
s = Solver()

# Add a biconditional
s.add(P == Q)

# Add a fact
s.add(P)

# Check satisfiability: s.check() should return sat
print(s.check())  # Should return 'sat'

# Inspect the model: s.model() should show P and Q both true
print(s.model())  # Should show P = True, Q = True

# Implement the z3_entails function using push/pop and verify that after adding P==Q and P, the solver entails Q and print the results.
def z3_entails(solver, query):
    """Check whether the solver's current assertions entail query."""
    solver.push()
    solver.add(Not(query))
    result = solver.check() == unsat
    solver.pop()
    return result


sat
[Q = True, P = True]
